# U3-02 · Medidas y tipos de agrupamiento: mayoristas y clientes RFM

*Machine Learning aplicado usando Python (IFCD093PO) · Unidad 3, apartados 3.1 a 3.3*

Ejecuta cada celda con **Mayús + Intro**. Todo funciona en el navegador; la primera vez tarda unos segundos en cargar scikit-learn.

> Datos (UCI, licencia CC BY 4.0), junto a este cuaderno; su preparación se explica en `LEEME.md`:
> Cardoso, M. (2013). *Wholesale customers*. https://doi.org/10.24432/C5030X ·
> Chen, D. (2012). *Online Retail II*. https://doi.org/10.24432/C5CG6D

## 1. Medidas de rendimiento con los mayoristas (3.2)

In [ ]:
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
    calinski_harabasz_score, adjusted_rand_score)

m = pd.read_csv("mayoristas.csv")
gasto = m.drop(columns=["canal", "region"])
X = StandardScaler().fit_transform(np.log1p(gasto))

def agrupar(X, k, semilla=0):
    return KMeans(k, n_init=10, random_state=semilla).fit_predict(X)

if __name__ == "__main__":
    print(gasto.describe().round(0).T[["mean", "50%", "max"]])
    for k in range(2, 7):
        e = agrupar(X, k)
        print(k, round(silhouette_score(X, e), 3),
              round(davies_bouldin_score(X, e), 2),
              round(calinski_harabasz_score(X, e), 1))
    e2 = agrupar(X, 2)
    print(pd.crosstab(e2, m.canal))
    print("ARI con el canal:",
          round(adjusted_rand_score(m.canal, e2), 3))
    # el mismo k-means sin logaritmo
    Xb = StandardScaler().fit_transform(gasto)
    eb = agrupar(Xb, 2)
    print("sin log:", np.bincount(eb),
          round(silhouette_score(Xb, eb), 3),
          round(adjusted_rand_score(m.canal, eb), 3))
    # estabilidad: reagrupar submuestras del 80 %
    rng = np.random.default_rng(1)
    for k in (2, 3, 4):
        ref, ari = agrupar(X, k), []
        for _ in range(20):
            i = rng.choice(len(X), int(.8 * len(X)), replace=False)
            ari.append(adjusted_rand_score(
                ref[i], agrupar(X[i], k, semilla=7)))
        print(k, "estabilidad", round(np.mean(ari), 3),
              round(np.min(ari), 3))

**Prueba.** Cambia `np.log1p(gasto)` por `gasto` en la línea que calcula `X` y vuelve a ejecutar la celda. ¿Qué número de grupos recomiendan ahora las medidas internas?

## 2. Segmentar a los clientes con RFM (3.3)

La última celda tarda unos segundos: ajusta siete mezclas gaussianas y dibuja el dendrograma.

In [ ]:
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score

c = pd.read_csv("clientes_rfm.csv", index_col="cliente")
Z = StandardScaler().fit_transform(np.log1p(c))

def perfil(etiquetas):
    p = c.groupby(etiquetas).median()
    p["clientes"] = pd.Series(etiquetas).value_counts()
    return p.round(0)

if __name__ == "__main__":
    km = KMeans(4, n_init=10, random_state=0).fit_predict(Z)
    print(perfil(km))
    je = AgglomerativeClustering(4, linkage="ward").fit_predict(Z)
    print(perfil(je))
    print("ARI k-medias/jerárquico:",
          round(adjusted_rand_score(km, je), 3))
    for eps in (0.2, 0.3, 0.5):
        db = DBSCAN(eps=eps, min_samples=10).fit_predict(Z)
        print(eps, "grupos", db.max() + 1,
              "ruido", (db == -1).sum())
    print(perfil(db))                    # el último: eps = 0,5
    print("una sola compra:", (c.frecuencia == 1).sum())
    for k in range(2, 9):
        g = GaussianMixture(k, n_init=3, random_state=0).fit(Z)
        print(k, "BIC", round(g.bic(Z)))
    import matplotlib.pyplot as plt
    from scipy.cluster.hierarchy import linkage, dendrogram
    L = linkage(Z, method="ward")
    plt.figure(figsize=(4.4, 2.0))
    dendrogram(L, truncate_mode="lastp", p=30, no_labels=True,
               color_threshold=L[-3, 2],  # corte: 4 grupos
               above_threshold_color="black")
    plt.ylabel("distancia de Ward"); plt.show()  # plt.savefig("dendro.pdf")

**Prueba.** Repite DBSCAN solo con los clientes que tienen más de una compra (`Z[c.frecuencia.values > 1]`). ¿Cambian los grupos?

## 3. Mezclas gaussianas con los mayoristas (3.3)

In [ ]:
import numpy as np, pandas as pd
from sklearn.mixture import GaussianMixture

bic = {k: GaussianMixture(k, n_init=3, random_state=0)
          .fit(X).bic(X) for k in range(1, 8)}
print({k: round(v) for k, v in bic.items()})
g = GaussianMixture(3, n_init=3, random_state=0).fit(X)
p = g.predict_proba(X)
e = p.argmax(axis=1)
print(pd.crosstab(e, m.canal))
print("pesos", g.weights_.round(2))
print("dudosos (máx. < 0,8):", (p.max(axis=1) < .8).sum())
gasto = m.drop(columns=["canal", "region"])
print(gasto.groupby(e).median().round(0).T)

**Ejercicio.** Prueba `GaussianMixture(3, covariance_type="diag")`. ¿Qué número de componentes elige ahora el BIC?

> Rousseeuw, P. J. (1987). Silhouettes: a graphical aid to the interpretation and validation of cluster analysis. *Journal of Computational and Applied Mathematics*, 20, 53–65. https://doi.org/10.1016/0377-0427(87)90125-7